# Aula 08.3: Organizando um programa com funções

Nesta aula, vamos melhorar passo a passo um programa que simula um LED piscando de acordo com a distância medida por um sensor. O objetivo não é apenas fazer o programa funcionar, mas deixá-lo claro, reutilizável e fácil de testar.

> Cada versão funciona. Observe qual problema de organização motiva a versão seguinte.

## Como usar este notebook

1. Leia a regra do sistema.
2. Execute uma versão de cada vez.
3. Identifique repetições e responsabilidades misturadas.
4. Compare com a versão seguinte.
5. Ao final, altere as medições e teste o programa completo.

O notebook usa funções, escopo, condicionais, laços, listas e retornos estudados anteriormente.

## Objetivos da aula

Ao final, você deverá conseguir:

- identificar código repetido;
- separar um problema em tarefas menores;
- criar funções com uma responsabilidade clara;
- compor funções pequenas para realizar uma tarefa maior;
- reduzir dependências de variáveis globais;
- separar regras de decisão de ações;
- criar uma função principal que coordena o programa;
- simular entradas e saídas para testar sem hardware real.

## 1. O problema

Vamos simular um sensor ultrassônico e um LED de alerta. A regra será:

- distância menor que 20 cm: LED pisca rapidamente, com intervalo de 2 segundos;
- distância maior ou igual a 20 cm: LED pisca lentamente, com intervalo de 5 segundos.

Como não há hardware conectado, as ações serão representadas por mensagens. A espera também será apenas informada, sem atrasar o notebook.

### Entradas, processamento e saídas

| Parte | Neste programa |
|---|---|
| entrada | distância medida e pino do LED |
| processamento | escolher o intervalo |
| saída | acender, apagar e informar espera |

Essa separação ajudará a definir as funções.

## 2. Versão 1: tudo no mesmo bloco

A primeira versão coloca regra e ações diretamente na estrutura condicional.

In [ ]:
distancia = 12
pino_led = 8

if distancia < 20:
    print(f'LED {pino_led}: ACESO')
    print('Aguardar 2 segundos')
    print(f'LED {pino_led}: APAGADO')
    print('Aguardar 2 segundos')
else:
    print(f'LED {pino_led}: ACESO')
    print('Aguardar 5 segundos')
    print(f'LED {pino_led}: APAGADO')
    print('Aguardar 5 segundos')

### O que já funciona?

- a condição escolhe o intervalo correto;
- as ações aparecem na ordem esperada;
- o exemplo é fácil de executar.

### O que pode melhorar?

- as ações de acender e apagar estão misturadas à regra;
- quase todo o código dos dois caminhos é repetido;
- mudar a forma de controlar o LED exigiria editar vários lugares.

## 3. Versão 2: funções para ações básicas

Vamos dar nomes às ações de acender, apagar e esperar.

In [ ]:
def acender_led(pino):
    print(f'LED {pino}: ACESO')

def apagar_led(pino):
    print(f'LED {pino}: APAGADO')

def esperar(tempo):
    print(f'Aguardar {tempo} segundos')

In [ ]:
distancia = 35
pino_led = 8

if distancia < 20:
    acender_led(pino_led)
    esperar(2)
    apagar_led(pino_led)
    esperar(2)
else:
    acender_led(pino_led)
    esperar(5)
    apagar_led(pino_led)
    esperar(5)

### O que melhorou?

- cada ação possui um nome;
- detalhes da simulação ficam dentro das funções;
- o bloco principal fica mais próximo da linguagem do problema.

Ainda repetimos a sequência acender, esperar, apagar e esperar nos dois caminhos.

## 4. Versão 3: compondo uma função maior

Uma função pode chamar outras funções. `piscar_led()` reúne a sequência que representa um ciclo completo.

In [ ]:
def piscar_led(pino, intervalo):
    acender_led(pino)
    esperar(intervalo)
    apagar_led(pino)
    esperar(intervalo)

In [ ]:
distancia = 12
pino_led = 8

if distancia < 20:
    piscar_led(pino_led, 2)
else:
    piscar_led(pino_led, 5)

### O que melhorou?

- a sequência repetida existe em apenas um lugar;
- a condição muda somente o intervalo;
- se a sequência do piscar mudar, editamos apenas `piscar_led()`.

Esse processo de melhorar a estrutura sem mudar o comportamento é chamado de **refatoração**.

## 5. Versão 4: separando a regra de decisão

A escolha do intervalo é uma regra do sistema. Podemos representá-la com uma função que apenas calcula e retorna um valor.

In [ ]:
def calcular_intervalo(distancia):
    if distancia < 20:
        return 2
    return 5

print(f'12 cm -> {calcular_intervalo(12)} segundos')
print(f'20 cm -> {calcular_intervalo(20)} segundos')
print(f'35 cm -> {calcular_intervalo(35)} segundos')

A função é fácil de testar porque não usa `input()`, não espera e não controla o LED. Para qualquer distância, ela apenas retorna o intervalo.

In [ ]:
distancia = 35
pino_led = 8
intervalo = calcular_intervalo(distancia)

piscar_led(pino_led, intervalo)

A regra decide **qual intervalo** usar. A função de ação decide **como piscar**. Cada uma tem uma responsabilidade.

## 6. Versão 5: processando uma medição

Agora reunimos a regra e a ação em uma função que processa uma única medição. Os dados chegam por parâmetros, sem depender de variáveis globais.

In [ ]:
def processar_medicao(distancia, pino_led):
    intervalo = calcular_intervalo(distancia)
    print(f'Distância medida: {distancia} cm')
    print(f'Intervalo escolhido: {intervalo} segundos')
    piscar_led(pino_led, intervalo)

processar_medicao(18, 8)

`processar_medicao()` coordena duas tarefas menores, mas não precisa conhecer de onde vieram a distância e o pino. Isso facilita usar dados reais ou simulados.

## 7. Versão final: função principal

A função `principal()` representa o fluxo de alto nível. Ela obtém os dados simulados e processa cada medição.

In [ ]:
def obter_dados_simulados():
    medicoes = [12, 35, 18, 25]
    pino_led = 8
    return medicoes, pino_led

A função retorna dois valores. Na chamada, eles serão desempacotados em `medicoes` e `pino_led`.

In [ ]:
def principal():
    medicoes, pino_led = obter_dados_simulados()

    for distancia in medicoes:
        print('--- Nova medição ---')
        processar_medicao(distancia, pino_led)

principal()

### Fluxo da versão final

```text
principal
  ├─ obter_dados_simulados
  └─ para cada distância: processar_medicao
       ├─ calcular_intervalo
       └─ piscar_led
            ├─ acender_led
            ├─ esperar
            ├─ apagar_led
            └─ esperar
```

A função principal permite entender o programa sem ler imediatamente todos os detalhes.

## 8. Comparando as versões

| Versão | Organização | Limitação principal |
|---|---|---|
| 1 | tudo no `if/else` | repetição e responsabilidades misturadas |
| 2 | ações básicas em funções | sequência de piscar repetida |
| 3 | função composta `piscar_led` | regra ainda dentro do fluxo |
| 4 | regra em `calcular_intervalo` | processa apenas um caso solto |
| 5 | função para uma medição | ainda falta coordenar várias entradas |
| final | dados, regra, ação e fluxo separados | adequada para a simulação proposta |

## 9. Funções com responsabilidade clara

Uma boa pergunta é: **qual tarefa esta função realiza?** A resposta deve ser curta e específica.

- `calcular_intervalo`: decide o intervalo;
- `acender_led`: acende o LED;
- `piscar_led`: executa um ciclo;
- `processar_medicao`: coordena uma medição;
- `principal`: coordena o programa.

Funções pequenas não são um objetivo isolado. A divisão é útil quando os nomes e as responsabilidades tornam o programa mais compreensível.

## 10. Dados por parâmetros, resultados por retorno

A versão final evita depender de variáveis globais para distância e pino. Cada função recebe o que precisa. Funções de cálculo retornam resultados; funções de saída mostram ações.

In [ ]:
def calcular_intervalos(medicoes):
    intervalos = []
    for distancia in medicoes:
        intervalos.append(calcular_intervalo(distancia))
    return intervalos

medicoes_teste = [10, 20, 30]
print(calcular_intervalos(medicoes_teste))

Como `calcular_intervalos()` retorna uma lista, podemos testá-la sem analisar mensagens na tela. Para `[10, 20, 30]`, o resultado esperado é `[2, 5, 5]`.

## 11. Simulação e hardware real

Durante o aprendizado, simular hardware com mensagens traz algumas vantagens:

- o notebook funciona em qualquer computador;
- não há espera real entre as ações;
- podemos repetir testes rapidamente;
- erros não afetam componentes físicos.

Em um projeto real, `acender_led()`, `apagar_led()` e `esperar()` poderiam usar uma biblioteca de hardware. As regras e a organização do restante do programa poderiam permanecer parecidas.

### Sobre `sleep()`

A função `sleep()` do módulo `time` pausa o programa:

```python
from time import sleep
sleep(2)
```

Ela não é usada nas células executáveis porque várias pausas tornariam a aula lenta. A função simulada `esperar()` mostra a intenção sem bloquear.

## 12. Evitando laços infinitos durante testes

Em sistemas embarcados, `while True` pode representar um dispositivo que funciona continuamente. Em um notebook, esse laço impediria o término da célula.

Para testar com segurança, usamos uma lista finita de medições ou um número máximo de ciclos.

In [ ]:
def executar_ciclos(distancia, pino_led, quantidade_de_ciclos):
    for ciclo in range(1, quantidade_de_ciclos + 1):
        print(f'--- Ciclo {ciclo} ---')
        processar_medicao(distancia, pino_led)

executar_ciclos(distancia=15, pino_led=8, quantidade_de_ciclos=2)

O limite permite observar o comportamento sem precisar interromper manualmente a execução.

## 13. Testando partes pequenas

Quando a regra está isolada, podemos conferir casos importantes com `assert`. Se a expressão for falsa, Python sinaliza uma falha. Se estiver correta, a célula termina normalmente.

In [ ]:
assert calcular_intervalo(0) == 2
assert calcular_intervalo(19) == 2
assert calcular_intervalo(20) == 5
assert calcular_intervalo(100) == 5

print('Todos os testes de intervalo passaram.')

Os valores `19` e `20` verificam exatamente a fronteira da regra. Testar limites ajuda a encontrar erros em `<`, `<=`, `>` e `>=`.

## 14. Exemplo adicional: alarme por temperatura

A mesma organização pode ser aplicada a outro problema.

In [ ]:
def classificar_temperatura(temperatura):
    if temperatura >= 35:
        return 'alerta alto'
    elif temperatura >= 25:
        return 'atenção'
    return 'normal'

def mostrar_alerta(temperatura, classificacao):
    print(f'{temperatura} °C: {classificacao}')

def processar_temperaturas(temperaturas):
    for temperatura in temperaturas:
        classificacao = classificar_temperatura(temperatura)
        mostrar_alerta(temperatura, classificacao)

processar_temperaturas([22, 28, 36])

A regra `classificar_temperatura()` retorna um valor e pode ser testada separadamente. `mostrar_alerta()` cuida da saída. `processar_temperaturas()` coordena a lista.

### Erros comuns de organização

| Situação | Problema | Possível melhoria |
|---|---|---|
| copiar o mesmo bloco em vários caminhos | alterações precisam ser repetidas | extraia uma função |
| função faz entrada, cálculo, espera e saída | muitas responsabilidades misturadas | separe tarefas |
| função depende de várias globais | comportamento fica escondido | use parâmetros |
| função apenas mostra um cálculo necessário depois | resultado não pode ser reutilizado | use `return` |
| nomes como `f1`, `coisa` ou `dados2` | intenção fica obscura | escolha verbo e objeto claros |
| `while True` em uma célula de teste | execução não termina | limite ciclos ou use lista finita |
| `sleep()` longo durante testes | validação fica lenta | simule a espera |
| criar funções para cada linha sem ganho de significado | fluxo fica fragmentado | agrupe uma responsabilidade real |

## 15. Prática guiada

Altere o limite, os intervalos e as medições. Observe que a mudança da regra fica concentrada em uma função.

In [ ]:
def escolher_intervalo(distancia, limite=25, rapido=1, lento=4):
    if distancia < limite:
        return rapido
    return lento

medicoes = [10, 24, 25, 40]
for distancia in medicoes:
    intervalo = escolher_intervalo(distancia)
    print(f'{distancia} cm -> {intervalo} segundo(s)')

## 16. Exercícios

### Nível 1: compreender a refatoração

1. Liste as responsabilidades de `acender_led`, `apagar_led`, `esperar`, `piscar_led`, `calcular_intervalo`, `processar_medicao` e `principal`.
2. Explique qual repetição foi removida pela função `piscar_led()`.
3. Explique por que `calcular_intervalo()` é mais fácil de testar do que o programa inteiro.
4. Altere a regra para usar intervalo 1 quando a distância for menor que 15 e intervalo 4 nos demais casos.
5. Teste os limites 14 e 15 com `assert`.

In [ ]:
# Resolva aqui os exercícios do Nível 1.



### Nível 2: separar responsabilidades

6. Crie `classificar_nota(nota)`, que apenas retorne `'aprovado'` ou `'reprovado'`.
7. Crie `mostrar_resultado(nome, nota, classificacao)`, que mostre uma mensagem.
8. Crie `processar_estudantes(estudantes)`, que receba um dicionário de nomes e notas e coordene as duas funções anteriores.
9. Teste com pelo menos três estudantes.
10. Escreva dois `assert` para verificar o limite de aprovação.

In [ ]:
# Resolva aqui os exercícios do Nível 2.



### Nível 3: refatorar um programa

O código abaixo mistura todas as etapas:

```python
precos = [10, 20, 30]
taxa = 0.1
total = 0
for preco in precos:
    total += preco
total = total * (1 + taxa)
print(f'Total: {total}')
```

11. Crie `calcular_subtotal(precos)`.
12. Crie `aplicar_taxa(subtotal, taxa)`.
13. Crie `mostrar_total(total)`.
14. Crie `processar_compra(precos, taxa)` para coordenar as funções.
15. Confirme com `assert` que `[10, 20, 30]` e taxa `0.1` resultam em `66.0`.

In [ ]:
# Resolva aqui os exercícios do Nível 3.



### Desafio: sensor de estacionamento

Crie um programa organizado com estas funções:

- `classificar_distancia(distancia)`: retorna `'perigo'` abaixo de 10 cm, `'atenção'` abaixo de 30 cm e `'livre'` nos demais casos;
- `escolher_intervalo(classificacao)`: retorna `1`, `3` ou `0` conforme a classificação;
- `emitir_alerta(classificacao, intervalo)`: mostra a situação e, quando o intervalo for maior que zero, simula um alerta;
- `processar_distancia(distancia)`: coordena classificação e alerta;
- `principal(medicoes)`: percorre uma lista finita de medições.

Teste as fronteiras `9`, `10`, `29` e `30` com `assert` e execute uma simulação completa.

In [ ]:
# Resolva o desafio aqui.



## 17. Resumo da aula

- Refatorar melhora a estrutura sem alterar o comportamento desejado.
- Código repetido pode indicar uma função ausente.
- Funções devem representar responsabilidades claras.
- Funções pequenas podem chamar outras e compor tarefas maiores.
- Regras de cálculo são mais fáceis de testar quando separadas das saídas.
- Parâmetros reduzem dependências globais.
- Uma função principal mostra o fluxo de alto nível.
- Simulações finitas tornam testes de hardware seguros e rápidos.

### Autoavaliação

Sem consultar o início, desenhe o fluxo da versão final e explique por que `calcular_intervalo()` não deve acender o LED.

## 18. Gabarito sugerido

> Consulte esta seção somente depois de tentar resolver os exercícios.

### Gabarito do Nível 1

In [ ]:
def calcular_intervalo_exercicio(distancia):
    if distancia < 15:
        return 1
    return 4

assert calcular_intervalo_exercicio(14) == 1
assert calcular_intervalo_exercicio(15) == 4

print('Regra alterada e limites validados.')

### Gabarito do Nível 2

In [ ]:
def classificar_nota(nota):
    if nota >= 7:
        return 'aprovado'
    return 'reprovado'

def mostrar_resultado(nome, nota, classificacao):
    print(f'{nome}: nota {nota} - {classificacao}')

def processar_estudantes(estudantes):
    for nome, nota in estudantes.items():
        classificacao = classificar_nota(nota)
        mostrar_resultado(nome, nota, classificacao)

assert classificar_nota(6.9) == 'reprovado'
assert classificar_nota(7.0) == 'aprovado'

processar_estudantes({'Ana': 8.0, 'Bruno': 6.5, 'Carla': 7.0})

### Gabarito do Nível 3

In [ ]:
def calcular_subtotal(precos):
    return sum(precos)

def aplicar_taxa(subtotal, taxa):
    return subtotal * (1 + taxa)

def mostrar_total(total):
    print(f'Total: R$ {total:.2f}')

def processar_compra(precos, taxa):
    subtotal = calcular_subtotal(precos)
    total = aplicar_taxa(subtotal, taxa)
    mostrar_total(total)
    return total

total = processar_compra([10, 20, 30], 0.1)
assert total == 66.0

### Uma solução para o desafio

In [ ]:
def classificar_distancia(distancia):
    if distancia < 10:
        return 'perigo'
    elif distancia < 30:
        return 'atenção'
    return 'livre'

def escolher_intervalo_alerta(classificacao):
    if classificacao == 'perigo':
        return 1
    elif classificacao == 'atenção':
        return 3
    return 0

def emitir_alerta(classificacao, intervalo):
    print(f'Situação: {classificacao}')
    if intervalo > 0:
        print(f'Alerta simulado a cada {intervalo} segundo(s)')
    else:
        print('Sem alerta sonoro')

def processar_distancia(distancia):
    print(f'Distância: {distancia} cm')
    classificacao = classificar_distancia(distancia)
    intervalo = escolher_intervalo_alerta(classificacao)
    emitir_alerta(classificacao, intervalo)

def principal_estacionamento(medicoes):
    for distancia in medicoes:
        print('--- Nova medição ---')
        processar_distancia(distancia)

assert classificar_distancia(9) == 'perigo'
assert classificar_distancia(10) == 'atenção'
assert classificar_distancia(29) == 'atenção'
assert classificar_distancia(30) == 'livre'

principal_estacionamento([7, 18, 35])

## Para continuar praticando

Escolha um exercício antigo que tenha entrada, cálculo e saída no mesmo bloco. Separe as responsabilidades em funções e crie pelo menos dois `assert` para testar a parte de cálculo.